# 03 - Built-in Tavily Search Agent

## Learning objective

Replace the custom Tavily wrapper with LangChain’s ready-made `TavilySearch` integration and compare the two approaches.

```text
User request -> Gemini -> built-in TavilySearch -> Gemini -> final answer
```

This notebook breaks the implementation into small, inspectable steps.

> **Quota note:** The final agent cell can make multiple Gemini and Tavily requests.


## 1. Check the active Python kernel

The executable should point to the Conda environment used for notebooks, such as `nlp_env`.


In [1]:
import sys

print("Python executable:", sys.executable)
print("Python version:", sys.version)


Python executable: C:\Users\inkk1r04\AppData\Local\anaconda3\envs\nlp_env\python.exe
Python version: 3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]


## 2. Optional package installation

If the course packages are not installed in the active kernel, remove the `#` and run the cell once. Then use **Kernel -> Restart Kernel**.


In [ ]:
# !pip install -r "../langchain-course/requirements.txt"


## 3. Corporate SSL and environment variables

`truststore` uses the Windows certificate store. The existing `.env` is loaded from the sibling script folder without printing secret values.


In [2]:
import os
from pathlib import Path

import truststore
from dotenv import load_dotenv

truststore.inject_into_ssl()

ENV_PATH = Path("../langchain-course/.env").resolve()
load_dotenv(dotenv_path=ENV_PATH)

print(".env exists:", ENV_PATH.exists())
print("Google key loaded:", bool(os.getenv("GOOGLE_API_KEY")))
print("Tavily key loaded:", bool(os.getenv("TAVILY_API_KEY")))
print("LangSmith key loaded:", bool(os.getenv("LANGSMITH_API_KEY")))
print("LangSmith tracing:", os.getenv("LANGSMITH_TRACING"))


.env exists: True
Google key loaded: True
Tavily key loaded: True
LangSmith key loaded: True
LangSmith tracing: true


## 4. Import the agent components

The important change from Notebook 02 is the import of `TavilySearch`. No custom `@tool` function or direct `TavilyClient` is needed.


In [3]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_tavily import TavilySearch

print("Imports completed successfully.")


Imports completed successfully.


## 5. Create the built-in Tavily tool

`max_results=5` limits the number of results returned by one search execution. Creating the object does not call Tavily.


In [5]:
from IPython.display import display

In [6]:
search = TavilySearch(max_results=5)

print("Tool type:", type(search))
print("Tool name:", search.name)
print("Tool description:", search.description)
print("Tool inputs:", search.args)


Tool type: <class 'langchain_tavily.tavily_search.TavilySearch'>
Tool name: tavily_search


'Tool description:'

'A search engine optimized for comprehensive, accurate, and trusted results. Useful for when you need to answer questions about current events. It not only retrieves URLs and snippets, but offers advanced search depths, domain management, time range filters, and image search, this tool delivers real-time, accurate, and citation-backed results.Input should be a search query.'

Tool inputs: {'query': {'description': 'Search query to look up', 'title': 'Query', 'type': 'string'}, 'include_domains': {'anyOf': [{'items': {'type': 'string'}, 'type': 'array'}, {'type': 'null'}], 'default': [], 'description': 'A list of domains to restrict search results to.\n\n        Use this parameter when:\n        1. The user explicitly requests information from specific websites (e.g., "Find climate data from nasa.gov")\n        2. The user mentions an organization or company without specifying the domain (e.g., "Find information about iPhones from Apple")\n\n        In both cases, you should determine the appropriate domains (e.g., ["nasa.gov"] or ["apple.com"]) and set this parameter.\n\n        Results will ONLY come from the specified domains - no other sources will be included.\n        Default is None (no domain restriction).\n        ', 'title': 'Include Domains'}, 'exclude_domains': {'anyOf': [{'items': {'type': 'string'}, 'type': 'array'}, {'type': 'null'}], 'default

In [7]:
display(search.description)

'A search engine optimized for comprehensive, accurate, and trusted results. Useful for when you need to answer questions about current events. It not only retrieves URLs and snippets, but offers advanced search depths, domain management, time range filters, and image search, this tool delivers real-time, accurate, and citation-backed results.Input should be a search query.'

## 6. Inspect the built-in tool schema

LangChain provides the name, description, argument model, execution logic, and Tavily API integration.


In [12]:
search.args_schema.model_json_schema()


{'additionalProperties': True,
 'description': 'Input for [TavilySearch]',
 'properties': {'query': {'description': 'Search query to look up',
   'title': 'Query',
   'type': 'string'},
  'include_domains': {'anyOf': [{'items': {'type': 'string'}, 'type': 'array'},
    {'type': 'null'}],
   'default': [],
   'description': 'A list of domains to restrict search results to.\n\n        Use this parameter when:\n        1. The user explicitly requests information from specific websites (e.g., "Find climate data from nasa.gov")\n        2. The user mentions an organization or company without specifying the domain (e.g., "Find information about iPhones from Apple")\n\n        In both cases, you should determine the appropriate domains (e.g., ["nasa.gov"] or ["apple.com"]) and set this parameter.\n\n        Results will ONLY come from the specified domains - no other sources will be included.\n        Default is None (no domain restriction).\n        ',
   'title': 'Include Domains'},
  'excl

## 7. Optional direct tool test

> **This cell makes one Tavily request but does not call Gemini.**


In [13]:
direct_result = search.invoke(
    {"query": "LangChain agents documentation"}
)

print(type(direct_result))
print(str(direct_result)[:1500])


<class 'dict'>
{'query': 'LangChain agents documentation', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://reference.langchain.com/python/langchain/agents', 'title': 'agents | langchain | LangChain Reference', 'content': "LangChain Reference home page\n\n⌘I\n\n## LangChain Assistant\n\nAsk a question to get started\n\n`Enter` to send•`Shift+Enter` new line\n\n## Menu\n\nOverviewAgentsMiddlewareModelsMessagesToolsEmbeddings\n\nMCP Adapters\n\nStandard Tests\n\nText Splitters\n\nLanguage\n\nTheme\n\n# Agents\n\nReference docs\n\nThis page contains reference documentation for Agents. See the docs for conceptual guides, tutorials, and examples on using Agents.\n\n## Classes\n\nClass\n\n### AgentState\n\nState schema for the agent.Class [...] ### ModelRequest\n\nModel request information for the agent.Class\n\n### ToolStrategy\n\nUse a tool calling strategy for model responses.Class\n\n### ProviderStrategy\n\nUse the model provider's native structured

In [14]:
direct_result.keys()

dict_keys(['query', 'follow_up_questions', 'answer', 'images', 'results', 'response_time', 'request_id'])

In [20]:
type(direct_result.get('results'))

list

In [21]:
direct_result.get('results')[0]

{'url': 'https://reference.langchain.com/python/langchain/agents',
 'title': 'agents | langchain | LangChain Reference',
 'content': "LangChain Reference home page\n\n⌘I\n\n## LangChain Assistant\n\nAsk a question to get started\n\n`Enter` to send•`Shift+Enter` new line\n\n## Menu\n\nOverviewAgentsMiddlewareModelsMessagesToolsEmbeddings\n\nMCP Adapters\n\nStandard Tests\n\nText Splitters\n\nLanguage\n\nTheme\n\n# Agents\n\nReference docs\n\nThis page contains reference documentation for Agents. See the docs for conceptual guides, tutorials, and examples on using Agents.\n\n## Classes\n\nClass\n\n### AgentState\n\nState schema for the agent.Class [...] ### ModelRequest\n\nModel request information for the agent.Class\n\n### ToolStrategy\n\nUse a tool calling strategy for model responses.Class\n\n### ProviderStrategy\n\nUse the model provider's native structured output method.Class\n\n### AutoStrategy\n\nAutomatically select the best strategy for structured output.\n\n## Functions\n\nFun

## 8. Initialize Gemini and create the agent

`create_agent()` receives the built-in tool exactly as it received the custom tool in Notebook 02.


In [22]:
MODEL = "gemini-3.5-flash-lite" #"gemini-3.6-flash"

llm = ChatGoogleGenerativeAI(
    model=MODEL,
    max_retries=2,
)

tools = [search]
agent = create_agent(model=llm, tools=tools)

print("Model:", MODEL)
print("Agent type:", type(agent))
print("Available tools:", [tool.name for tool in tools])


Model: gemini-3.5-flash-lite
Agent type: <class 'langgraph.graph.state.CompiledStateGraph'>
Available tools: ['tavily_search']


## 9. Build the input without making an API call


In [23]:
question = (
    "Search for 3 job postings for an AI engineer "
    "using LangChain in the Bay Area on LinkedIn "
    "and list their details."
)

agent_input = {
    "messages": [HumanMessage(content=question)]
}

print(agent_input)


{'messages': [HumanMessage(content='Search for 3 job postings for an AI engineer using LangChain in the Bay Area on LinkedIn and list their details.', additional_kwargs={}, response_metadata={})]}


## 10. Run the agent

> **This cell can make several Gemini and Tavily requests and create one LangSmith trace.**


In [24]:
result = agent.invoke(agent_input)

print("Agent execution completed.")


Agent execution completed.


In [25]:
type(result)

dict

In [26]:
result.keys()

dict_keys(['messages'])

In [28]:
type(result.get('messages'))

list

In [29]:
len(result.get('messages'))

8

In [45]:
result.get('messages')[1]

AIMessage(content=[], additional_kwargs={'function_call': {'name': 'tavily_search', 'arguments': '{"query": "AI engineer LangChain jobs Bay Area LinkedIn"}'}, '__gemini_function_call_thought_signatures__': {'call_679998': 'El4KXAFpFH0T6sBQc4zI61NAxT8WpZWkAJo5cTkR/j0MKK45qb5bTJ3K1Es6tHCsBT009+Aeh34pNiqMz6bclwOfzbjPL2tyMlsMoCFdA+4ZhinL0rQruIF+9vuJa8U5'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0c838-1e8e-7ac2-a12e-c9041aa5a482-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'AI engineer LangChain jobs Bay Area LinkedIn'}, 'id': 'call_679998', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1515, 'output_tokens': 25, 'total_tokens': 1540, 'input_token_details': {'cache_read': 0}})

In [43]:
result.get('messages')[1].tool_calls

[{'name': 'tavily_search',
  'args': {'query': 'AI engineer LangChain jobs Bay Area LinkedIn'},
  'id': 'call_679998',
  'type': 'tool_call'}]

In [46]:
result.get('messages')[7]

AIMessage(content=[{'type': 'text', 'text': 'Here are 3 detailed job postings for AI / Software Engineers utilizing LangChain in the San Francisco Bay Area, sourced from LinkedIn and professional tech networks:\n\n---\n\n### 1. **Fullstack Software Engineer, Applied AI**\n* **Company:** LangChain\n* **Location:** San Francisco Bay Area (Hybrid / On-site)\n* **Compensation:** $170,000 – $240,000 / year\n* **Role Overview:** LangChain is hiring full-stack Applied AI Engineers to help build next-generation AI agents and production-ready applications. \n* **Key Responsibilities & Tech Stack:**\n  * Building scalable AI agent architectures.\n  * Utilizing the core LangChain ecosystem (LangChain, LangGraph, LangSmith) alongside full-stack tools.\n  * Translating product requirements into reliable production code that interfaces with modern Large Language Models (LLMs).\n\n---\n\n### 2. **Deployed Engineer, Professional Services**\n* **Company:** LangChain\n* **Location:** San Francisco, CA (

In [44]:
result.get('messages')[7].tool_calls

[]

## 11. Inspect the accumulated state and final answer


In [47]:
print("Result keys:", result.keys())
print("Message count:", len(result["messages"]))

for index, message in enumerate(result["messages"]):
    print("=" * 70)
    print(index, type(message).__name__)
    tool_calls = getattr(message, "tool_calls", None)
    if tool_calls:
        print("Tool calls:", tool_calls)
    else:
        print(str(message.content)[:800])


Result keys: dict_keys(['messages'])
Message count: 8
0 HumanMessage
Search for 3 job postings for an AI engineer using LangChain in the Bay Area on LinkedIn and list their details.
1 AIMessage
Tool calls: [{'name': 'tavily_search', 'args': {'query': 'AI engineer LangChain jobs Bay Area LinkedIn'}, 'id': 'call_679998', 'type': 'tool_call'}]
2 ToolMessage
{"query": "AI engineer LangChain jobs Bay Area LinkedIn", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.linkedin.com/jobs/view/fullstack-software-engineer-applied-ai-at-langchain-4381519355", "title": "Fullstack Software Engineer, Applied AI at LangChain - Jobs", "content": "We're hiring fullstack Applied AI Engineers to help us build AI agents that ... San Francisco Bay Area $170,000 - $240,000 3 months ago. Fullstack / AI", "score": 0.77751994, "raw_content": null, "id": "38116a-00"}, {"url": "https://www.linkedin.com/jobs/langchain-jobs-san-francisco-bay-area", "title": "616 Langchain jo

In [48]:
final_message = result["messages"][-1]
print(final_message.text)


Here are 3 detailed job postings for AI / Software Engineers utilizing LangChain in the San Francisco Bay Area, sourced from LinkedIn and professional tech networks:

---

### 1. **Fullstack Software Engineer, Applied AI**
* **Company:** LangChain
* **Location:** San Francisco Bay Area (Hybrid / On-site)
* **Compensation:** $170,000 – $240,000 / year
* **Role Overview:** LangChain is hiring full-stack Applied AI Engineers to help build next-generation AI agents and production-ready applications. 
* **Key Responsibilities & Tech Stack:**
  * Building scalable AI agent architectures.
  * Utilizing the core LangChain ecosystem (LangChain, LangGraph, LangSmith) alongside full-stack tools.
  * Translating product requirements into reliable production code that interfaces with modern Large Language Models (LLMs).

---

### 2. **Deployed Engineer, Professional Services**
* **Company:** LangChain
* **Location:** San Francisco, CA (Hybrid)
* **Role Overview:** A customer-facing engineering role

## 12. Custom tool versus built-in tool

### Custom tool

- You write and maintain the wrapper function.
- You control logging and result transformation.
- You must accurately describe inputs and outputs.

### Built-in tool

- Less code.
- Integration package supplies schema and execution logic.
- Easier for standard use cases.

The agent’s conceptual loop remains unchanged.

## Practice exercises

- Change `max_results` and compare output size.
- Inspect the built-in tool description.
- Compare the schemas from Notebooks 02 and 03.
- Ask a question that does not require current web information and observe whether the model still searches.
